# SuspiciousLogin Dataset — Empirical Threat Patterns (Article 2)

Temporal, geographic, and authentication patterns in suspicious vs normal logins, with formal statistical tests (chi-square, Cramér's V, odds ratios with 95% confidence intervals, Mann-Whitney U, Mann-Kendall trend test) rather than descriptive comparison alone.

Runs against the **public** schema only.

**A caution repeated from Article 1's limitation notice**: `label` is Google's own operational suspicion signal, not a confirmed compromise. A statistically significant association here means the FEATURE predicts Google's flag, not that the feature causes or confirms an actual attack. This matters especially for the geographic results below -- do not read "logins from country X are more often flagged" as "logins from country X are malicious": most of this dataset's own normal traffic also comes from a wide mix of countries.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import pymannkendall as mk

PUBLIC_FILE = Path("data/processed/suspicious_logins_public_v1.csv")
if not PUBLIC_FILE.exists():
    PUBLIC_FILE = Path("suspicious_logins_public_v1.csv")

df = pd.read_csv(PUBLIC_FILE)
print(f"Loaded {len(df)} rows")
print(f"Overall positive rate: {df['label'].mean()*100:.2f}%")

## 2. Statistical test helper functions

Defined once, reused throughout -- every categorical comparison below
uses the same chi-square + Cramér's V pair, every binary comparison
uses the same odds-ratio + 95% CI, so results are directly comparable
to each other.

In [ ]:
def chi_square_and_cramers_v(df, feature_col, label_col="label", min_support=10):
    """Chi-square test of independence between a categorical feature
    and the label, plus Cramer's V as an effect-size measure (chi-square
    alone grows with sample size and doesn't indicate how STRONG an
    association is). Categories with fewer than min_support total rows
    are dropped first -- a chi-square test on a near-empty cell is not
    reliable."""
    counts = df[feature_col].value_counts()
    keep = counts[counts >= min_support].index
    sub = df[df[feature_col].isin(keep)]
    table = pd.crosstab(sub[feature_col], sub[label_col])
    chi2, p_value, dof, expected = stats.chi2_contingency(table)
    n = table.values.sum()
    min_dim = min(table.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else float("nan")
    return {"chi2": chi2, "p_value": p_value, "dof": dof,
           "cramers_v": cramers_v, "n": n, "categories_kept": len(keep)}


def odds_ratio_with_ci(df, binary_feature_col, label_col="label"):
    """Odds ratio for a binary feature vs the binary label, with a 95%
    CI via the standard log-odds-ratio approach
    (Woolf's method) -- the standard, simple approach for a 2x2 table,
    adequate here since none of this dataset's cells are anywhere near
    zero."""
    table = pd.crosstab(df[binary_feature_col], df[label_col])
    a = table.loc[1, 1] if (1 in table.index and 1 in table.columns) else 0
    b = table.loc[1, 0] if (1 in table.index and 0 in table.columns) else 0
    c = table.loc[0, 1] if (0 in table.index and 1 in table.columns) else 0
    d = table.loc[0, 0] if (0 in table.index and 0 in table.columns) else 0
    odds_ratio = (a * d) / (b * c) if b * c > 0 else float("inf")
    se_log_or = np.sqrt(1/max(a,1) + 1/max(b,1) + 1/max(c,1) + 1/max(d,1))
    log_or = np.log(odds_ratio) if odds_ratio not in (0, float("inf")) else float("nan")
    ci_low = np.exp(log_or - 1.96 * se_log_or)
    ci_high = np.exp(log_or + 1.96 * se_log_or)
    return {"odds_ratio": odds_ratio, "ci_low": ci_low, "ci_high": ci_high,
           "a": a, "b": b, "c": c, "d": d}


def shannon_entropy(counts):
    probabilities = counts / counts.sum()
    return -np.sum(probabilities * np.log2(probabilities.replace(0, np.nan).fillna(1)))

print("Helper functions defined.")

## 3. Temporal patterns

In [ ]:
hourly = df.groupby("event_hour")["label"].agg(["mean", "count"])
hourly["mean"] *= 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(hourly.index, hourly["mean"])
ax.axhline(df["label"].mean()*100, linestyle="--", color="gray", label="Overall rate")
ax.set_xlabel("Hour of day (UTC)")
ax.set_ylabel("Suspicious rate (%)")
ax.set_title("Suspicious rate by hour of day")
ax.legend()
plt.tight_layout()
plt.savefig("temporal_hourly_rate.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
heatmap_data = df.groupby(["day_of_week", "event_hour"])["label"].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(24))
ax.set_xticklabels(range(24))
ax.set_yticks(range(7))
ax.set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
ax.set_xlabel("Hour of day (UTC)")
ax.set_title("Suspicious rate (%) by day of week x hour")
plt.colorbar(im, ax=ax, label="Suspicious rate (%)")
plt.tight_layout()
plt.savefig("temporal_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
print("=== Nocturnal risk ratio (is_night_login) ===")
result = odds_ratio_with_ci(df, "is_night_login")
print(f"Odds ratio: {result['odds_ratio']:.3f} (95% CI: [{result['ci_low']:.3f}, {result['ci_high']:.3f}])")
print(f"n = {result['a']+result['b']+result['c']+result['d']}")
print()

print("=== Weekend risk ratio (is_weekend) ===")
result = odds_ratio_with_ci(df, "is_weekend")
print(f"Odds ratio: {result['odds_ratio']:.3f} (95% CI: [{result['ci_low']:.3f}, {result['ci_high']:.3f}])")

In [ ]:
monthly = df.groupby("month")["label"].mean() * 100
print("Suspicious rate by month:")
print(monthly.round(2))
print()

# Mann-Kendall: is there a statistically significant monotonic trend
# over time, not just visual impression? Confirms (or not) the
# declining-rate pattern already noted qualitatively around the
# Jul-Aug academic vacation period (see docs/RISK_SCORE_METHODOLOGY_EN.md).
trend_result = mk.original_test(monthly.sort_index().values)
print(f"Mann-Kendall trend test: {trend_result.trend} "
     f"(p={trend_result.p:.4f}, tau={trend_result.Tau:.3f})")

## 4. Geographic patterns

**A striking case worth flagging explicitly**: a handful of countries
with real, non-trivial event counts (Argentina, Chile, Colombia,
Mexico -- 293 to 603 events each) show a 100% suspicious rate,
confirmed present already in the raw extraction (not a processing
artifact). Investigated in detail: every event in each of these
countries comes from a DIFFERENT user and a DIFFERENT network (no
repeats), concentrated in a single major city per country, with varied
authentication methods -- not a bot-like pattern or a mass-compromise
signature. This is consistent with each of these being a genuinely
distinct person's first-ever login from a country the institution's
traffic almost never includes (dominated by Angola and Brazil) --
Google's own detection appears to flag "country essentially unseen for
this institution" close to deterministically, an extreme version of
the `is_new_country` pattern already confirmed with a moderate odds
ratio elsewhere in this notebook. Reported here as a genuine finding,
not excluded or treated as an error -- but interpreted carefully: this
reflects institutional traffic novelty, not something about the
countries or people themselves.

In [ ]:
print("=== Suspicious rate by country (minimum 30 events) ===")
country_stats = df.groupby("ip_country")["label"].agg(["mean", "count"])
country_stats = country_stats[country_stats["count"] >= 30].sort_values("mean", ascending=False)
country_stats["mean"] = (country_stats["mean"] * 100).round(2)
print(country_stats)

full_rate = country_stats[country_stats["mean"] == 100.0]
if len(full_rate) > 0:
    print(f"\nCountries at exactly 100% suspicious rate (see notebook note above "
         f"for why this is a genuine, investigated finding, not excluded here):")
    print(list(full_rate.index))

In [ ]:
print("=== Country/continent change flags: odds ratios ===")
for col in ["is_new_country", "country_changed", "continent_changed", "multiple_country_logins_24h"]:
    result = odds_ratio_with_ci(df, col)
    print(f"{col}: OR={result['odds_ratio']:.3f} "
         f"(95% CI: [{result['ci_low']:.3f}, {result['ci_high']:.3f}])")

In [ ]:
# Shannon entropy of each user's country distribution -- a measure of
# how geographically scattered a user's own login history is. Computed
# per user (needs >= 2 events to be meaningful), then compared between
# users who were EVER flagged suspicious vs users who never were.
country_counts_per_user = df.groupby(["actor_pseudo_id", "ip_country"]).size().reset_index(name="n")
user_entropy = country_counts_per_user.groupby("actor_pseudo_id").apply(
    lambda g: shannon_entropy(g["n"]), include_groups=False)

user_ever_suspicious = df.groupby("actor_pseudo_id")["label"].max()
user_event_count = df.groupby("actor_pseudo_id").size()

entropy_df = pd.DataFrame({
    "entropy": user_entropy,
    "ever_suspicious": user_ever_suspicious,
    "n_events": user_event_count,
}).dropna()
entropy_df = entropy_df[entropy_df["n_events"] >= 2]  # entropy undefined/trivial for 1 event

print(f"Users with >=2 events (entropy is meaningful): {len(entropy_df)}")
print(entropy_df.groupby("ever_suspicious")["entropy"].describe())

u_stat, p_value = stats.mannwhitneyu(
    entropy_df[entropy_df["ever_suspicious"]==1]["entropy"],
    entropy_df[entropy_df["ever_suspicious"]==0]["entropy"],
    alternative="two-sided")
print(f"\nMann-Whitney U test: U={u_stat:.1f}, p={p_value:.4f}")

## 5. Authentication patterns

In [ ]:
print("=== login_type: frequency and suspicious rate ===")
login_type_stats = df.groupby("login_type")["label"].agg(["count", "mean"])
login_type_stats["mean"] = (login_type_stats["mean"] * 100).round(2)
print(login_type_stats.sort_values("count", ascending=False))
print()

test = chi_square_and_cramers_v(df, "login_type")
print(f"Chi-square test: chi2={test['chi2']:.2f}, p={test['p_value']:.6f}, "
     f"Cramer's V={test['cramers_v']:.4f} (n={test['n']})")

In [ ]:
print("=== Binary authentication/context features: odds ratios ===")
for col in ["is_business_hours", "new_ip", "impossible_travel",
           "multiple_country_logins_24h"]:
    result = odds_ratio_with_ci(df, col)
    print(f"{col}: OR={result['odds_ratio']:.3f} "
         f"(95% CI: [{result['ci_low']:.3f}, {result['ci_high']:.3f}])")

## 6. Numeric features vs label: Mann-Whitney U tests

In [ ]:
numeric_features_to_test = [
    "distinct_ips_7d", "distinct_ips_30d", "logins_24h", "logins_7d", "logins_30d",
    "avg_logins_per_day", "hours_since_last_login", "days_since_first_login",
    "distinct_users_per_network_24h", "travel_speed_kmh",
]

print(f"{'feature':<32} {'median (label=0)':>18} {'median (label=1)':>18} {'p-value':>12}")
for col in numeric_features_to_test:
    group0 = df[df["label"]==0][col].dropna()
    group1 = df[df["label"]==1][col].dropna()
    u_stat, p_value = stats.mannwhitneyu(group1, group0, alternative="two-sided")
    print(f"{col:<32} {group0.median():>18.2f} {group1.median():>18.2f} {p_value:>12.6f}")

## 7. Interpretable logistic regression summary

In [ ]:
import statsmodels.api as sm

logit_features = ["is_night_login", "is_weekend", "is_new_country", "country_changed",
                  "new_ip", "impossible_travel", "multiple_country_logins_24h"]
X = df[logit_features].fillna(0).astype(float)
X = sm.add_constant(X)
y = df["label"]

model = sm.Logit(y, X).fit(disp=0)
print(model.summary())
print()
print("Odds ratios (exp(coefficient)):")
print(np.exp(model.params).round(3))

## 8. Summary table

In [ ]:
summary_rows = []
for col in ["is_night_login", "is_weekend", "is_new_country", "country_changed",
           "continent_changed", "multiple_country_logins_24h", "new_ip",
           "impossible_travel", "is_business_hours"]:
    r = odds_ratio_with_ci(df, col)
    summary_rows.append({"feature": col, "odds_ratio": round(r["odds_ratio"], 3),
                         "ci_low": round(r["ci_low"], 3), "ci_high": round(r["ci_high"], 3)})

summary = pd.DataFrame(summary_rows)
summary.to_csv("article2_odds_ratios.csv", index=False)
summary